# 09 · Multitarefa: uma ResNet-152 para entrada/saída e distância do disparo — mesmo protocolo do P1

**Sessão P2 · ambiente L4.** Uma só rede, com o extrator compartilhado e duas cabeças: **entrada/saída** (2 classes, todas as imagens) e **distância do disparo** (encostado, curta, distância; só nas entradas). Mesma receita, mesmas dobras e mesmo código do `02` e do `04`: ResNet-152 ImageNet → Dropout 0,5 → {cabeça E/S, cabeça distância}; 2 fases (15 + 45 épocas); última época; AUC por probabilidade. A perda é L = L_E/S + L_dist (λ = 1); as saídas não entram em L_dist (`ignore_index = −1`), exatamente como o protocolo já faz. As redes de tarefa única (ST) são as do P1 e não rodam de novo.

| config | cabeças | parâmetros | par ST do P1 (2 redes) |
|---|---|---|---|
| `mt_kan` | KAN `[2048,128,k]`, grade 10, ordem 3 (= cabeça d) | 66.017.728 | 124.161.536 (−47 %) |
| `mt_mlp` | MLP `[2048,128,k]` (= cabeça b) | 58.668.997 | 116.812.805 (−50 %) |

Sementes 42, 43 e 44 (as mesmas das ST do P1), na ordem semente → config → dobra.

**O que mudar na célula de configuração:**
* `TESTE_RAPIDO = True` na 1ª vez, **só na aba A** (`09_multitarefa_entrada_saida_distancia.ipynb`): as duas configs rodam 2 + 2 épocas na dobra 0, há um teste de matar e retomar (comparado ao ruído da própria GPU) e a medição do tempo de inferência (1 forward da MT × 2 do par ST), ~15 min. Depois mude para `False` e **Executar tudo**.
* `SO_CONFIGS`: `["mt_kan"]` na aba A e `["mt_mlp"]` na aba B (`09_multitarefa_abaB.ipynb`), as duas ao mesmo tempo (~2,3 h cada). Com `None`, uma aba só roda as duas, uma depois da outra (~4,5 h).

**Se a sessão cair:** abra de novo e **Executar tudo**: unidades prontas são puladas e a do meio retoma da última época salva. **Uma aba, um arquivo:** nunca abra o mesmo notebook em duas abas.

Resultados em `resultados/multitarefa/`: por unidade, `metricas.json`, `pred_teste.csv` e `historico.csv`; por config, `runs.csv`, `resumo.json` e `oof.csv`; `calibracao_mt.json` (tempo de inferência) e `teste_rapido_ok.json`. Se a perda virar NaN, a fila para e grava `falha_nan.json` na pasta da unidade.

In [12]:
# ==== Configuração desta sessão ====
DIR_DRIVE = "/content/drive/MyDrive/Colab Notebooks/tcc-final"  # pasta tcc-final (conta padrão, authuser=0)
RAIZ_TRABALHO = "/content"   # onde o dataset é clonado e o cache é gerado (disco local, temporário)
SHA_PROTOCOLO_ESPERADO = "b7cd9baca53edd96c855cca10914fc258efa8ddfd0937cc0d3e9572de5ef9e13"
SHA_PLUGIN_ESPERADO = "b9841e0f8c0192bf67df96f318e384659a96659c5eda08737dae740f612c1814"   # multitarefa.py
SHA_DOBRAS_ESPERADO = "ccef0fdcb8e98d3e94dbf5373eafed01936487b41eb8c41102f04970dc551f58"   # as dobras do P1
ABA = "A"                    # dono das travas desta aba (a cópia da 2ª aba usa "B")
PERMITIR_RESERVA = False     # True só para rodar na T4 como reserva (marcado com † na tabela)
SO_CONFIGS = ["mt_kan"]      # aba A: ["mt_kan"]; aba B: ["mt_mlp"]; None = as duas nesta aba
FILTRO_SEMENTES = None       # ex.: [42] para rodar só uma semente
TESTE_RAPIDO = False          # 1ª vez, na aba A: True; depois False
CKPT_INTERVALO_MIN = None    # None = usa a recomendação do 01
DESLIGAR_AO_FIM = True       # desconecta o ambiente quando a rodada termina

In [13]:
# Dependências fixadas (rode ANTES de importar qualquer coisa da albumentations)
!pip install -q "albumentations==2.0.8" "git+https://github.com/Blealtan/efficient-kan@7b6ce1c87f18c8bc90c208f6b494042344216b11"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [14]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import sys, os, json, time
from pathlib import Path
import numpy as np, pandas as pd, torch
sys.path.insert(0, DIR_DRIVE)
import protocolo_comum as pc

pc.checar_sentinela_drive(DIR_DRIVE)   # PARA aqui se o Colab estiver logado em outra conta
assert pc.sha256_protocolo() == SHA_PROTOCOLO_ESPERADO, (
    "O protocolo_comum.py do Drive não é o publicado. Não rode nada; avise o P1.")
ARQ_PLUGIN = f"{DIR_DRIVE}/multitarefa.py"
assert pc.sha256_arquivo(ARQ_PLUGIN) == SHA_PLUGIN_ESPERADO, (
    "O multitarefa.py do Drive não é o publicado. Não rode nada; avise a sessão P2.")
pc.carregar_plugin(ARQ_PLUGIN)          # registra mt_kan e mt_mlp (o SHA vai para cada metricas.json)
mt = sys.modules["multitarefa"]
print("Dependências:", pc.checar_dependencias())
print("Protocolo", pc.PROTOCOLO_VERSAO, "| SHA-256", pc.sha256_protocolo()[:16], "… | plugin", SHA_PLUGIN_ESPERADO[:16], "…")
GPU = pc.exigir_gpu(permitir_reserva=PERMITIR_RESERVA)   # rodadas comparáveis só na L4
pc.definir_id_sessao(ABA)
print("GPU:", GPU["nome"], "| comparável:", GPU["comparavel"], "| dono das travas:", pc.ID_SESSAO)

Dependências: {'albumentations': '2.0.8', 'efficient_kan_commit': '7b6ce1c87f18c8bc90c208f6b494042344216b11'}
Protocolo 1.0.0 | SHA-256 b7cd9baca53edd96 … | plugin b9841e0f8c0192bf …
GPU: NVIDIA L4 | comparável: True | dono das travas: aba-A


In [16]:
# Clona o dataset e o leguwoi nos commits fixados e gera o cache 224×224 só nesta sessão (~3–5 min)
sess = pc.preparar_sessao(
    RAIZ_TRABALHO,
    caminho_dobras=f"{DIR_DRIVE}/dados/{pc.ARQUIVO_DOBRAS}",
    caminho_meta=f"{DIR_DRIVE}/dados/{pc.ARQUIVO_META_DOBRAS}",
)
assert sess.meta["dobras_sha256"] == SHA_DOBRAS_ESPERADO, "O CSV de dobras não é o das rodadas do P1. Pare e avise a sessão P2."
R, C = f"{DIR_DRIVE}/resultados", f"{DIR_DRIVE}/checkpoints"
RT, CT = f"{DIR_DRIVE}/resultados_teste", f"{DIR_DRIVE}/checkpoints_teste"
print(json.dumps(sess.meta, indent=1))
if sess.meta.get("cache_confere") is False:
    print("⚠️ O cache diverge do registrado no 00 (versão do OpenCV?). Divergência:", sess.meta.get("cache_divergencia"))
# 2.554 imagens, 486 casos, nenhum caso em 2 dobras, ≥ 2 casos de encostado por dobra, saída a ±3 pp;
# teste e pesos de classe da distância iguais aos da rede de tarefa única do P1, dobra a dobra
print("Dobras (as mesmas das redes de tarefa única do P1):")
display(mt.checar_dados(sess.df))

{
 "dataset_commit": "f6793899b081a5d6de34100498d9345a7db4c951",
 "leguwoi_commit": "83d82f1f696f6a8d7e2969388ed026274f873ffa",
 "dobras_sha256": "ccef0fdcb8e98d3e94dbf5373eafed01936487b41eb8c41102f04970dc551f58",
 "semente_dobras": 42,
 "cache_s": 8.0,
 "cache_sha256": "fc5fbdf6d3a90913cc8ab2153f495e1b0f1c7074585011bb91266a4326f562c1",
 "cache_sha256_esperado": "fc5fbdf6d3a90913cc8ab2153f495e1b0f1c7074585011bb91266a4326f562c1",
 "cache_confere": true
}
Dobras (as mesmas das redes de tarefa única do P1):


,dobra,n_treino,entradas_treino,n_teste,entradas_teste,teste_encostado,teste_curta,teste_distancia,casos_encostado_teste,frac_saida_teste,pesos_es,pesos_dist
0,0,2048,1523,506,360,4,53,303,2,0.2885,"[0.672, 1.95]","[13.36, 2.836, 0.389]"
1,1,2095,1542,459,341,10,25,306,4,0.2571,"[0.679, 1.894]","[16.062, 2.483, 0.394]"
2,2,2023,1485,531,398,13,43,342,5,0.2505,"[0.681, 1.88]","[17.069, 2.619, 0.391]"
3,3,2016,1477,538,406,11,61,334,3,0.2454,"[0.682, 1.87]","[15.882, 2.879, 0.386]"
4,4,2034,1505,520,378,4,50,324,2,0.2731,"[0.676, 1.922]","[13.202, 2.756, 0.39]"


In [17]:
cal = json.loads((Path(R) / "calibracao_l4.json").read_text())
PREC = cal.get("decisao_precisao")
assert PREC in ("amp", "fp32"), "Rode o 01 (ponte fp32 × AMP) antes."
CKPT = cal.get("recomendacao_ckpt_intervalo_min", 0.0) if CKPT_INTERVALO_MIN is None else CKPT_INTERVALO_MIN
UC_H = (cal.get("unidades_por_hora") or {}).get("valor") or 1.54
EXP = mt.EXPERIMENTO
DIR_EXP = Path(R) / EXP
ARQ_OK = DIR_EXP / "teste_rapido_ok.json"
ARQ_BENCH = DIR_EXP / "calibracao_mt.json"
configs = list(SO_CONFIGS) if SO_CONFIGS else list(mt.CONFIGS.values())
sementes = [s for s in mt.SEMENTES if not FILTRO_SEMENTES or s in FILTRO_SEMENTES]
fila = mt.montar_fila(sementes, configs)
smoke = cal.get("multitarefa_smoke") or {}
if not smoke.get("ok"):
    print("⚠️ O teste da multitarefa da parte 7 do 01 ainda não rodou (ou falhou). Rode o 01 antes da rodada longa.")

MIN_PADRAO = 8.8   # min por dobra: ~8,4 da rede E/S do P1 na L4 + a 2ª cabeça
def min_por_unidade():
    r = pc.ler_resultados(R, EXP)
    if len(r):
        r = r[~r["teste_rapido"].astype(bool) & r["gpu"].fillna("").str.contains("L4")]
    if len(r) == 0:
        return MIN_PADRAO, "estimativa"
    return float(r["treino_min"].mean()) + 0.7, f"média das {len(r)} unidades prontas"

def previsao(u):
    m, _ = min_por_unidade()
    return f"▶ {u['config']} s{u['semente']} dobra {u['dobra']}: previsão ~{m:.1f} min ≈ {m / 60 * UC_H:.2f} UC — {pc.agora_iso()}"

faltam = [u for u in fila if not (mt.dir_unidade(R, u) / "metricas.json").exists()]
m, fonte = min_por_unidade()
print(f"Experimento {EXP} | aba {ABA} | configs {configs} | sementes {sementes} | precisão {PREC} | checkpoint a cada ≥ {CKPT} min")
print(f"{len(fila)} unidades nesta aba, {len(faltam)} a fazer (ordem: semente → config → dobra):")
print(pd.DataFrame(fila).groupby(["semente", "config"], sort=False).size().to_string())
print(f"\nPrevisão do que falta: ~{len(faltam) * m / 60:.1f} h de L4 ≈ {len(faltam) * m / 60 * UC_H:.1f} UC "
      f"({m:.1f} min por dobra, {fonte}; + ~5 min de preparo por sessão)")

Experimento multitarefa | aba A | configs ['mt_kan'] | sementes [42, 43, 44] | precisão amp | checkpoint a cada ≥ 5.0 min
15 unidades nesta aba, 15 a fazer (ordem: semente → config → dobra):
semente  config
42       mt_kan    5
43       mt_kan    5
44       mt_kan    5

Previsão do que falta: ~2.2 h de L4 ≈ 3.4 UC (8.8 min por dobra, estimativa; + ~5 min de preparo por sessão)


## Teste rápido (só na aba A: as duas configs, 2 + 2 épocas na dobra 0; matar e retomar; tempo de inferência; pasta de teste)

In [18]:
if TESTE_RAPIDO:
    import shutil
    EXP_T = "t09_rapido"
    for e in (EXP_T, f"{EXP_T}_ref", f"{EXP_T}_ref2", f"{EXP_T}_int"):
        shutil.rmtree(f"{RT}/{e}", ignore_errors=True); shutil.rmtree(f"{CT}/{e}", ignore_errors=True)
    problemas = []
    te0 = sess.df[sess.df["fold"] == 0]
    tr0 = sess.df[sess.df["fold"] != 0]
    # 1) as duas configs, dobra 0 inteira, 2 + 2 épocas
    for cfg in mt.CONFIGS.values():
        u = {"experimento": EXP_T, "config": cfg, "semente": 42, "dobra": 0, "coluna_dobra": "fold"}
        t0 = time.time()
        mt.executar_fila_mt(sess, [u], RT, CT, precisao=PREC, limite_epocas=(2, 2), espera_trava_s=0)
        d = mt.dir_unidade(RT, u)
        r = json.loads((d / "metricas.json").read_text())
        h, p = pd.read_csv(d / "historico.csv"), pd.read_csv(d / "pred_teste.csv", dtype={"case_id": str})
        for t in ("es", "dist"):
            mm = r["metricas"][t]
            print(f"{cfg} {t}: acc {mm['acc']:.3f} | AUC {mm['auc']:.3f} | F1 {mm['f1_macro']:.3f} | {mm['n']} imagens na métrica")
        print(f"   perda na última época: E/S {h['perda_es'].iloc[-1]:.4f} | distância {h['perda_dist'].iloc[-1]:.4f} | "
              f"{r['parametros']['treinaveis_fase2']:,} parâmetros | {time.time() - t0:.0f} s")
        problemas += mt.checar_unidade(r)
        if r["n_teste"] != len(te0) or len(p) != len(te0):
            problemas.append(f"{cfg}: o teste deveria ser a dobra 0 inteira ({len(te0)} imagens)")
        if r["metricas"]["dist"]["n"] != int((te0["y_es"] == 0).sum()):
            problemas.append(f"{cfg}: a métrica de distância deveria usar só as {int((te0['y_es'] == 0).sum())} entradas do teste")
        if p[[f"p_dist_{c}" for c in range(3)]].isna().any().any():
            problemas.append(f"{cfg}: a distância deveria ser prevista em todas as imagens de teste (métrica de pipeline)")
        if not np.isfinite(h[["perda", "perda_es", "perda_dist"]].to_numpy()).all():
            problemas.append(f"{cfg}: perda não finita no histórico")
        if not np.allclose(r["pesos_classe"]["dist"], pc.pesos_classe(tr0["y_dist"].to_numpy(), 3)):
            problemas.append(f"{cfg}: pesos de classe da distância não contam só as entradas do treino")
        if r["ambiente"]["protocolo_sha256"] != SHA_PROTOCOLO_ESPERADO: problemas.append("SHA do protocolo")
        if r["plugins"].get("multitarefa", {}).get("sha256") != SHA_PLUGIN_ESPERADO: problemas.append("SHA do plugin")
        if r["dados"].get("dobras_sha256") != SHA_DOBRAS_ESPERADO: problemas.append("SHA das dobras")
        if (r["ambiente"]["gpu"] or {}).get("nome") != GPU["nome"]: problemas.append("GPU não registrada")
    # 2) matar e retomar: para no meio da F1 e no meio da F2. Uma 2ª rodada SEM interrupção mede o ruído da própria GPU:
    #    na F2 (backbone treinando em AMP) duas rodadas iguais já diferem um pouco. A retomada tem de ficar dentro desse
    #    ruído (3× ele, com piso de 1e-3) e a F1 (só as cabeças) tem de sair igual.
    torch.backends.cudnn.deterministic = True
    kw = dict(precisao=PREC, limite_epocas=(2, 2), limite_imagens=256, espera_trava_s=0)
    u0 = {"config": mt.CONFIGS["kan"], "semente": 42, "dobra": 0, "coluna_dobra": "fold"}
    u_ref, u_ref2, u_int = ({**u0, "experimento": f"{EXP_T}_{s}"} for s in ("ref", "ref2", "int"))
    mt.executar_fila_mt(sess, [u_ref], RT, CT, **kw)
    mt.executar_fila_mt(sess, [u_ref2], RT, CT, **kw)
    for parar in (1, 3):
        try:
            mt.executar_fila_mt(sess, [u_int], RT, CT, parar_apos_epoca=parar, **kw)
        except pc.InterrupcaoSimulada as e:
            print("interrupção simulada:", e)
    mt.executar_fila_mt(sess, [u_int], RT, CT, **kw)
    torch.backends.cudnn.deterministic = False
    ch, cp = ["perda", "perda_es", "perda_dist"], ["p_es_1"] + [f"p_dist_{c}" for c in range(3)]

    def diferencas(ua, ub):
        da, db = mt.dir_unidade(RT, ua), mt.dir_unidade(RT, ub)
        h1, h2 = pd.read_csv(da / "historico.csv"), pd.read_csv(db / "historico.csv")
        p1, p2 = pd.read_csv(da / "pred_teste.csv", dtype={"case_id": str}), pd.read_csv(db / "pred_teste.csv", dtype={"case_id": str})
        f1 = (h1["fase"] == 1).to_numpy()
        return {"n": [len(h1), len(h2)],
                "f1": float(np.abs(h1.loc[f1, ch].to_numpy() - h2.loc[f1, ch].to_numpy()).max()),
                "perda": float(np.abs(h1[ch].to_numpy() - h2[ch].to_numpy()).max()),
                "prob": float(np.abs(p1[cp].to_numpy() - p2[cp].to_numpy()).max())}

    ret, ruido = diferencas(u_ref, u_int), diferencas(u_ref, u_ref2)
    lim = {"f1": max(1e-5, 3 * ruido["f1"]), "perda": max(1e-3, 3 * ruido["perda"]), "prob": max(1e-3, 3 * ruido["prob"])}
    print(f"ruído da GPU (2 rodadas sem interrupção): F1 {ruido['f1']:.1e} | perdas {ruido['perda']:.1e} | probabilidades {ruido['prob']:.1e}")
    print(f"retomada (com 2 interrupções):           F1 {ret['f1']:.1e} | perdas {ret['perda']:.1e} | probabilidades {ret['prob']:.1e}")
    print(f"limites:                                 F1 {lim['f1']:.1e} | perdas {lim['perda']:.1e} | probabilidades {lim['prob']:.1e}")
    dif_perda, dif_prob = ret["perda"], ret["prob"]
    if ret["n"] != [4, 4] or ruido["n"] != [4, 4] or any(ret[k] > lim[k] for k in lim):
        problemas.append("a retomada não reproduziu o treino (diferença acima do ruído da GPU)")
    for u in (u_ref, u_ref2, u_int):
        if (Path(CT) / u["experimento"]).exists() and any((Path(CT) / u["experimento"]).rglob("ckpt_last.pt")):
            problemas.append(f"checkpoint não apagado ao fim da unidade ({u['experimento']})")
    # 3) nada de imagem, cache ou checkpoint nas pastas de resultados
    proib = [x for base in (Path(RT), DIR_EXP) if base.exists() for x in base.rglob("*")
             if x.suffix.lower() in (".npy", ".jpg", ".jpeg", ".png", ".pt")]
    if proib: problemas.append(f"arquivos proibidos nos resultados: {proib[:3]}")
    # 4) tempo de inferência para as duas saídas: 1 forward da MT × 2 forwards do par ST (lotes 1 e 32)
    bench = json.loads(ARQ_BENCH.read_text()) if ARQ_BENCH.exists() else {}
    if bench.get("plugin_sha256") != SHA_PLUGIN_ESPERADO or (bench.get("gpu") or {}).get("nome") != GPU["nome"]:
        t0 = time.time()
        bench = mt.benchmark_inferencia(precisao=PREC)
        bench["plugin_sha256"] = SHA_PLUGIN_ESPERADO
        pc.gravar_json_atomico(bench, ARQ_BENCH)
        print(f"tempo de inferência medido em {time.time() - t0:.0f} s")
    display(mt.tabela_benchmark(bench).round(3))
    print("Problemas:", problemas or "nenhum")
    assert not problemas, problemas
    pc.gravar_json_atomico({"ok": True, "quando": pc.agora_iso(), "plugin_sha256": SHA_PLUGIN_ESPERADO,
                            "protocolo_sha256": SHA_PROTOCOLO_ESPERADO, "gpu": GPU["nome"], "precisao": PREC,
                            "retomada": {"max_dif_perda": dif_perda, "max_dif_prob": dif_prob, "ruido_gpu": ruido, "limites": lim}}, ARQ_OK)
    print("✅ Teste rápido OK. ➡️ Agora mude TESTE_RAPIDO = False e clique em Executar tudo. "
          "Para a 2ª aba: 1 minuto depois, abra o 09_multitarefa_abaB.ipynb e clique em Executar tudo.")

## Rodada

In [19]:
assert not TESTE_RAPIDO, "TESTE_RAPIDO está True: mude para False para rodar a etapa."
ok = json.loads(ARQ_OK.read_text()) if ARQ_OK.exists() else {}
assert ok.get("ok") and ok.get("plugin_sha256") == SHA_PLUGIN_ESPERADO, (
    "O teste rápido ainda não passou com este multitarefa.py. Rode antes o 09 da aba A com TESTE_RAPIDO = True.")
fila_json = DIR_EXP / f"fila_aba{ABA}.json"
antiga = json.loads(fila_json.read_text()) if fila_json.exists() else []
pc.gravar_json_atomico(antiga + [u for u in fila if u not in antiga], fila_json)
feitos = mt.executar_fila_mt(sess, fila, R, C, previsao=previsao, precisao=PREC, ckpt_intervalo_min=CKPT)
print("Concluídas nesta sessão:", len(feitos))

▶ mt_kan s42 dobra 0: previsão ~8.8 min ≈ 0.23 UC — 2026-10-21T19:18:54+00:00
  mt_kan s42 d0 | época 01/60 (F1) | perda 1.6886 | acc treino 0.639 | 4.4s
  mt_kan s42 d0 | época 10/60 (F1) | perda 1.2674 | acc treino 0.734 | 4.3s
  mt_kan s42 d0 | época 15/60 (F1) | perda 1.2098 | acc treino 0.737 | 4.3s
  mt_kan s42 d0 | época 20/60 (F2) | perda 1.0699 | acc treino 0.766 | 10.0s
  mt_kan s42 d0 | época 30/60 (F2) | perda 0.8029 | acc treino 0.819 | 10.1s
  mt_kan s42 d0 | época 40/60 (F2) | perda 0.6845 | acc treino 0.833 | 10.0s
  mt_kan s42 d0 | época 50/60 (F2) | perda 0.7044 | acc treino 0.840 | 10.2s
  mt_kan s42 d0 | época 60/60 (F2) | perda 0.6891 | acc treino 0.851 | 9.9s
✅ mt_kan s42 d0: acc 0.8360 | AUC 0.8974 | F1 0.8045 | treino 8.6 min
▶ mt_kan s42 dobra 1: previsão ~9.3 min ≈ 0.24 UC — 2026-10-21T19:28:33+00:00
  mt_kan s42 d1 | época 01/60 (F1) | perda 1.7113 | acc treino 0.646 | 4.3s
  mt_kan s42 d1 | época 10/60 (F1) | perda 1.2554 | acc treino 0.729 | 4.3s
  mt_kan s

## Resultados desta aba (por config: runs.csv, resumo.json, oof.csv) e checagens de sanidade

In [20]:
METR = [f"{t}_{m}" for t in ("es", "dist") for m in ("acc", "prec_macro", "rec_macro", "f1_macro", "acc_balanceada", "auc")]

def agregar_config(cfg):
    # Grava runs.csv (1 linha = semente × dobra), resumo.json e oof.csv da config, em resultados/multitarefa/<config>/.
    runs = pc.ler_resultados(R, EXP)
    if runs.empty:
        print("Nada concluído ainda em", EXP)
        return runs
    runs = runs[(runs["config"] == cfg) & ~runs["teste_rapido"].astype(bool)].copy()
    if runs.empty:
        print("Nada concluído ainda em", cfg)
        return runs
    d = DIR_EXP / cfg
    runs.to_csv(d / "runs.csv", index=False)
    resumo = {}
    for sem, g in runs.groupby("semente"):
        resumo[f"s{sem}"] = {
            "n_dobras": int(len(g)),
            **{m: {"media": float(g[m].mean()), "dp": float(g[m].std(ddof=1)) if len(g) > 1 else None} for m in METR},
            "treino_min_por_dobra": float(g["treino_min"].mean()),
            "params_cabeca": int(g["params_cabeca"].iloc[0]),
            "params_total": int(g["params_total"].iloc[0]),
            "gpus": sorted(g["gpu"].dropna().unique().tolist()),
        }
    pc.gravar_json_atomico(resumo, d / "resumo.json")
    partes = [pd.read_csv(Path(r["caminho"]) / "pred_teste.csv", dtype={"case_id": str}).assign(config=r["config"], semente=r["semente"], dobra=r["dobra"])
              for _, r in runs.iterrows()]
    pd.concat(partes).to_csv(d / "oof.csv", index=False)
    print(f"{cfg}: {len(runs)} unidades -> runs.csv, resumo.json, oof.csv")
    return runs

for cfg in configs:
    runs = agregar_config(cfg)
    if runs.empty:
        continue
    display(runs.groupby("semente")[["es_acc", "es_auc", "es_f1_macro", "dist_acc", "dist_auc", "dist_f1_macro", "treino_min"]]
            .agg(["mean", "std", "count"]).round(4))
    # sanidade (plano P2): acc E/S por dobra entre 0,80 e 0,89; F1 macro da distância > 0,31 (prever sempre "distância" dá 0,307)
    fora = runs[(runs["es_acc"] < 0.80) | (runs["es_acc"] > 0.89)]
    baixa = runs[runs["dist_f1_macro"] <= 0.31]
    for _, x in fora.iterrows():
        print(f"⚠️ {cfg} s{x['semente']} d{x['dobra']}: acurácia E/S {x['es_acc']:.3f} fora de [0,80; 0,89]")
    for _, x in baixa.iterrows():
        print(f"⚠️ {cfg} s{x['semente']} d{x['dobra']}: F1 macro da distância {x['dist_f1_macro']:.3f} ≤ 0,31 (perto de prever sempre a classe majoritária)")
    if fora.empty and baixa.empty:
        print(f"{cfg}: checagens de sanidade OK em todas as {len(runs)} unidades")

mt_kan: 15 unidades -> runs.csv, resumo.json, oof.csv


es_acc                es_auc               es_f1_macro                \
           mean     std count    mean     std count        mean     std count   
semente                                                                         
42       0.8187  0.0171     5  0.8953  0.0123     5      0.7844  0.0215     5   
43       0.8225  0.0165     5  0.8950  0.0139     5      0.7866  0.0201     5   
44       0.8179  0.0237     5  0.8924  0.0153     5      0.7806  0.0274     5   

        dist_acc  ...       dist_auc               dist_f1_macro          \
            mean  ... count     mean     std count          mean     std   
semente           ...                                                      
42        0.8176  ...     5   0.8060  0.0657     5        0.4735  0.0632   
43        0.8335  ...     5   0.8005  0.0744     5        0.4941  0.0604   
44        0.8217  ...     5   0.8075  0.0570     5        0.4798  0.0548   

              treino_min                
        count       mean     std count  
semente                                 
42          5     8.5680  0.1199     5  
43          5     8.5955  0.1163     5  
44          5     8.5873  0.1067     5  

[3 rows x 21 columns]

⚠️ mt_kan s42 d3: acurácia E/S 0.792 fora de [0,80; 0,89]
⚠️ mt_kan s43 d3: acurácia E/S 0.797 fora de [0,80; 0,89]
⚠️ mt_kan s44 d2: acurácia E/S 0.793 fora de [0,80; 0,89]
⚠️ mt_kan s44 d3: acurácia E/S 0.796 fora de [0,80; 0,89]


⚠️ Se `DESLIGAR_AO_FIM = True`, a célula abaixo desconecta o ambiente (não gasta unidades parado). No teste rápido ela não chega a rodar: a célula da rodada para antes.

In [21]:
if DESLIGAR_AO_FIM:
    from google.colab import runtime
    runtime.unassign()